GEETA 4B — run a Jupyter notebook as a plain Python script.

Why: nbconvert/kernel execution was crashing the M3 assembly run (process
terminated, no traceback). Running the same cells top-to-bottom in a single
CPython process gives us a real traceback, real memory behaviour, and a
deterministic exit code.

Usage:
    jupyter execute tools/nb_run.ipynb notebooks/03_dataset_assembly.ipynb   # from project root
    jupyter execute tools/nb_run.ipynb --keep /tmp/nb03.py notebooks/03_...ipynb
Options:
    --keep PATH   keep the generated script at PATH (default: temp, deleted after)
    --timeout MS  wall-clock timeout in ms (default: none)

## Bootstrap

This notebook is a GEETA runtime module. Executed standalone (e.g. `jupyter execute tools/verifier.ipynb`),
the next cell registers the notebook-as-module loader so `geeta.*` imports resolve to `tools/*.ipynb`.
When imported as `geeta.<name>` via `tools/_nbmodule.ipynb`, the same cell runs (idempotent).


In [ ]:
# --- GEETA notebook-module bootstrap (idempotent; makes tools/*.ipynb importable as geeta.*) ---
import json as _json
import pathlib as _pathlib

_ROOT = _pathlib.Path.cwd()
if not (_ROOT / "tools" / "_nbmodule.ipynb").exists():      # run from project root or tools/
    _ROOT = _ROOT.parent
_nb = _json.loads((_ROOT / "tools" / "_nbmodule.ipynb").read_text())
_src = "".join("".join(c["source"]) for c in _nb["cells"] if c["cell_type"] == "code")
exec(compile(_src, str(_ROOT / "tools" / "_nbmodule.ipynb"), "exec"), globals())
bootstrap(_ROOT)  # noqa: E501


In [ ]:
import json
import pathlib
import subprocess
import sys
import tempfile

In [ ]:


def nb_to_script(nb_path: pathlib.Path) -> str:
    nb = json.loads(nb_path.read_text())
    parts = [f"# generated from {nb_path} by tools/nb_run.py — do not edit\n"]
    for i, cell in enumerate(nb.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue
        src = "".join(cell.get("source", []))
        if not src.strip():
            continue
        parts.append(f"# ============ cell {i} ============\n{src}\n")
    return "\n".join(parts)

In [ ]:


def main() -> int:
    args = [a for a in sys.argv[1:]]
    keep = None
    timeout_ms = None
    nbs = []
    i = 0
    while i < len(args):
        if args[i] == "--keep":
            keep = args[i + 1]; i += 2
        elif args[i] == "--timeout":
            timeout_ms = int(args[i + 1]); i += 2
        else:
            nbs.append(args[i]); i += 1
    if not nbs:
        print(__doc__)
        return 2

    # __file__ is undefined inside notebook cells, so resolve from it when
    # present (generated script) and fall back to the working directory.
    candidates = []
    if globals().get("__file__"):
        candidates.append(pathlib.Path(globals()["__file__"]).resolve().parent)
    candidates.append(pathlib.Path.cwd())
    py = sys.executable
    for base in candidates:
        for cand in (base, base.parent):
            vp = cand / ".venv" / "bin" / "python"
            if vp.exists():
                py = vp
                break
        if py != sys.executable:
            break

    rc = 0
    for nb in nbs:
        p = pathlib.Path(nb)
        script = nb_to_script(p)
        if keep:
            sp = pathlib.Path(keep)
            sp.write_text(script)
        else:
            with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
                f.write(script)
                sp = pathlib.Path(f.name)
        print(f"=== running {p} ({len(script)} chars) with {py} ===", flush=True)
        cmd = [str(py), str(sp)]
        proc = subprocess.run(cmd, cwd=str(p.parent.parent), timeout=timeout_ms / 1000 if timeout_ms else None)
        if not keep:
            sp.unlink(missing_ok=True)
        if proc.returncode != 0:
            print(f"!! {p} exited {proc.returncode}")
            rc = proc.returncode
            break
        print(f"=== {p} OK ===\n")
    return rc

In [ ]:


# Notebooks run cells with __name__ == '__main__' too, but sys.argv then
# holds ipykernel launch args — only run the CLI under plain CPython.
_in_kernel = any("ipykernel" in a for a in sys.argv[:2])
if __name__ == "__main__" and not _in_kernel:
    sys.exit(main())